# Figures 1 - Input Layers

Generates the study-area map, OD demand and observational/network input panels.

In [26]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

PALETTE = {
    "src_dest": "#E7D1D1",
    "rnd_rnd": "#DCC7DF",
    "rnd_dest": "#C4DADD",
    "src_rnd": "#C8DEC8",

    "roads_main": "#8D8D8D",
    "roads_minor": "#CFCFCF",

    "text": "#2F2F2F",
    "background": "#FAFAFA",
    "grid": "#DDDDDD",
}

paper_cmap = LinearSegmentedColormap.from_list(
    "paper_activity",
    [
        "#EAF6F7",  # very low
        "#AFCFD4",  # low
        "#D5C0DC",  # medium
        "#D9B8B8",  # high
        "#9E5F63",  # very high
    ],
    N=256,
)

# ---------- GLOBAL MATPLOTLIB STYLE ----------
plt.rcParams.update({

    # Figure
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["background"],
    "savefig.facecolor": PALETTE["background"],

    # Text
    "text.color": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],

    # Font sizes
    "font.size": 18,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 26,

    # Axes
    "axes.edgecolor": "#BDBDBD",
    "axes.linewidth": 1.2,
    "axes.grid": False,

    # Grid
    "grid.color": PALETTE["grid"],
    "grid.alpha": 0.4,
    "grid.linewidth": 0.8,

    # Legend
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#DDDDDD",

    # Lines
    "lines.linewidth": 2.2,

    # Ticks
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,

    # DPI
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # PDF/SVG publication quality
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

In [27]:
# Common imports and output utility
# Source: requested notebook standard; run from repository root.
try:
    import os
    from pathlib import Path
    import warnings
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.colors import LogNorm
    import seaborn as sns
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import contextily
    from PIL import Image

    start_dir = Path.cwd().resolve()
    ROOT = next((candidate for candidate in [start_dir, *start_dir.parents]
                 if (candidate / "article/tex/epjds_main_draft.tex").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Repository root not found from current working directory")
    os.chdir(ROOT)
    images_dir = Path("article/tex/images")
    images_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
    plt.rcParams.update({"savefig.dpi": 300, "pdf.fonttype": 42, "ps.fonttype": 42})

    def save_fig_local(fig, stem):
        # Write the manuscript target as both raster and vector output.
        stem = Path(stem).stem
        for ext in ("png", "pdf"):
            fig.savefig(images_dir / f"{stem}.{ext}", bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"OK: saved {stem}.png and {stem}.pdf in {images_dir}")

    print(f"OK: setup complete; repo root is {ROOT}; image output is {images_dir.resolve()}")
except Exception as exc:
    print(f"ERROR setup: {type(exc).__name__}: {exc}")
    raise


OK: setup complete; repo root is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep; image output is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep/article/tex/images


In [28]:
# 1a_map_study_area.png - lightweight required cartographic implementation.
# The repository contains several historical copies of the road network.
# Read candidates until the first usable LineString layer is found to avoid
# duplicating the full network in a very large publication PDF.
try:
    preferred_vectors = [
        Path("article/data/uniq/carto.geojson"),
        Path("MCF/data/raw_inputs/er-carto.geojson"),
        Path("work_carto/new_carto/carto.geojson"),
        Path("work_carto/rete_corretta_integrata_NOV22/rete_corretta_integrata_NOV22.geojson"),
    ]
    discovered = []
    for carto_dir in [Path("carto"), Path("vars/carto"), Path("work_carto")]:
        if carto_dir.exists():
            discovered.extend(
                p for p in carto_dir.rglob("*")
                if p.suffix.lower() in {".geojson", ".gpkg", ".shp"}
            )
    candidates = list(dict.fromkeys(preferred_vectors + sorted(discovered)))
    gdf_roads = None
    selected_layer = None
    for path in candidates:
        if not path.exists():
            continue
        try:
            layers = [None]
            if path.suffix.lower() == ".gpkg" and hasattr(gpd, "list_layers"):
                layers = list(gpd.list_layers(path)["name"])
            for layer in layers:
                frame = gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)
                line_mask = frame.geometry.geom_type.isin(["LineString", "MultiLineString"])
                if line_mask.any():
                    gdf_roads = frame.loc[line_mask, ["geometry"]].copy()
                    if gdf_roads.crs is None:
                        gdf_roads = gdf_roads.set_crs(4326)
                    gdf_roads = gdf_roads.to_crs(3857)
                    selected_layer = f"{path}" + (f":{layer}" if layer else "")
                    break
            if gdf_roads is not None:
                break
        except Exception as read_exc:
            print(f"INFO: skipped {path}: {read_exc}")
    if gdf_roads is None:
        raise FileNotFoundError("No line network found in cartographic candidates")

    fig, ax = plt.subplots(figsize=(10, 8.5), constrained_layout=True)
    gdf_roads.plot(ax=ax, linewidth=1, color=PALETTE["roads_main"], alpha=0.9, zorder=2)
    xmin, ymin, xmax, ymax = gdf_roads.total_bounds
    margin = 0.04 * max(xmax - xmin, ymax - ymin)
    ax.set_xlim(xmin - margin, xmax + margin)
    ax.set_ylim(ymin - margin, ymax + margin)
    try:
        contextily.add_basemap(ax, source=contextily.providers.CartoDB.Positron,crs=gdf_roads.crs, attribution_size=6)
    except Exception as tile_exc:
        print(f"INFO: basemap unavailable; saving vector map: {tile_exc}")
    scale_m = 50000
    x0, y0 = xmin + margin, ymin + margin
    ax.plot([x0, x0 + scale_m], [y0, y0], color=PALETTE["text"], lw=3, zorder=4)
    ax.text(x0 + scale_m / 2, y0 + margin * 0.25, "50 km",
            ha="center", va="bottom", fontsize=22)
    # ax.set_title("Emilia-Romagna road network and study area")
    ax.set_axis_off()
    save_fig_local(fig, "1a_map_study_area")
    plt.show()
    print(f"OK: selected {selected_layer}; {len(gdf_roads):,} road features")
except Exception as exc:
    print(f"ERROR 1a_map_study_area: {type(exc).__name__}: {exc}")


OK: saved 1a_map_study_area.png and 1a_map_study_area.pdf in article/tex/images
OK: selected article/data/uniq/carto.geojson; 4,186 road features


In [29]:
# 1b_od_demand.png - weekday activity by Emilia-Romagna municipality.
# Reconstructs article/results/od/activity_map_weekday.png from the original inputs.
try:
    source_root = ROOT.parent / "scaling_fundamental_diagram" / "dataset"
    trips_path = source_root / "ER" / "ER_data_raw.csv"
    shapes_path = source_root / "italia" / "italy_ace" / "italy_ace.shp"
    if not trips_path.exists() or not shapes_path.exists():
        raise FileNotFoundError("Required ER_data_raw.csv or italy_ace.shp input is missing")

    totals = []
    usecols = ["date", "time", "ACE_orig", "ACE_dest", "N"]
    for chunk in pd.read_csv(trips_path, usecols=usecols, chunksize=500_000):
        stamp = pd.to_datetime(chunk["date"] + " " + chunk["time"], errors="coerce")
        origin = chunk["ACE_orig"].astype(str).str.replace("C", "", regex=False)
        destination = chunk["ACE_dest"].astype(str).str.replace("C", "", regex=False)
        selected = (
            origin.str.startswith("0080")
            & destination.str.startswith("0080")
            & (origin != destination)
            & (stamp > pd.Timestamp("2020-01-01 00:00:00"))
            & (stamp <= pd.Timestamp("2020-01-31 23:00:00"))
            & (stamp.dt.dayofweek < 5)
        )
        totals.append(
            chunk.loc[selected].assign(city=origin[selected].str[:9]).groupby("city")["N"].sum()
        )
    weekday_activity = pd.concat(totals, axis=1).fillna(0).sum(axis=1).rename("N").reset_index()

    municipalities = gpd.read_file(shapes_path)[["UID", "geometry"]]
    municipalities["city"] = municipalities["UID"].astype(str).str[:9]
    municipalities = municipalities[municipalities["city"].str.startswith("0080")]
    municipalities = municipalities.dissolve(by="city", as_index=False)
    activity_map = municipalities.merge(weekday_activity, on="city", how="inner")
    positive = activity_map.loc[activity_map["N"] > 0, "N"]
    norm = LogNorm(
        vmin=np.percentile(positive, 2),
        vmax=np.percentile(positive, 98),
    )

    plt.rcParams.update({"font.size": 16})
    fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
    
    fig.patch.set_facecolor("#F5F5F5")
    ax.set_facecolor("#F5F5F5")
    
    activity_map.plot(
        column="N",
        cmap=paper_cmap,
        linewidth=0.9,
        edgecolor="#ECECEC",
        norm=norm,
        ax=ax,
    )    
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    sm = plt.cm.ScalarMappable(cmap=paper_cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=ax, shrink=0.35, pad=0.02,
        format=ticker.FuncFormatter(lambda val, pos: rf"$10^{{{int(np.log10(val))}}}$")
    )
    cbar.set_label("N")
    save_fig_local(fig, "1b_od_demand")
    print(f"OK: weekday activity map uses {len(activity_map)} municipalities")
except Exception as exc:
    print(f"ERROR 1b_od_demand: {type(exc).__name__}: {exc}")
    raise


OK: saved 1b_od_demand.png and 1b_od_demand.pdf in article/tex/images
OK: weekday activity map uses 344 municipalities


In [30]:
# 1c_coils_mapping.png - TODO/template.
# Candidate source: python/preanalysis/regione_shapefile.py:25-52 (ER_coils.png).
import contextily as ctx

try:
    roads = gdf_roads if "gdf_roads" in globals() else gpd.read_file("article/data/uniq/carto.geojson").to_crs(3857)
    point_files = list(Path(".").rglob("*MTS*.shp")) + list(Path(".").rglob("*coil*.geojson"))
    points = None
    for path in point_files:
        candidate = gpd.read_file(path)
        if not candidate.empty and candidate.geometry.geom_type.isin(["Point", "MultiPoint"]).any():
            points = candidate.to_crs(3857)
            break
    fig, ax = plt.subplots(figsize=(9, 7.5), dpi=300, constrained_layout=True)

    roads.plot(
        ax=ax,
        color=PALETTE["roads_main"],
        linewidth=0.45,
        alpha=0.75,
        zorder=2,
    )

    if points is None:
        ax.text(
            0.5, 0.03,
            "TODO: MTS point layer not located",
            transform=ax.transAxes,
            ha="center",
            fontsize=18,
            color="#D55E00",
            zorder=5,
        )
    else:
        points.plot(
            ax=ax,
            color=PALETTE["src_dest"],
            edgecolor=PALETTE["text"],
            linewidth=0.7,
            marker="o",
            markersize=55,
            alpha=0.95,
            label="MTS sensors",
            zorder=4,
        )

        ax.legend(
            fontsize=18,
            markerscale=1.5,
            frameon=True,
            facecolor="white",
            edgecolor="#DDDDDD",
            loc="lower right",
        )

    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        crs=roads.crs,
        attribution=False,
        zoom="auto",
        zorder=1,
    )

    ax.set_title(
        "Magnetic-coil sensor mapping",
        fontsize=24,
        color=PALETTE["text"],
        pad=14,
    )

    ax.set_axis_off()
    save_fig_local(fig, "1c_coils_mapping")
except Exception as exc:
    print(f"ERROR/TODO 1c_coils_mapping: provide MTS shapefile: {type(exc).__name__}: {exc}")


OK: saved 1c_coils_mapping.png and 1c_coils_mapping.pdf in article/tex/images


In [31]:
# 1d_osm_graph.png - observed daily profile at coil station 282.
# Keep the established output stem because it is already wired into the manuscript.
try:
    import matplotlib.dates as mdates

    coil_path = Path("data/coils_table_230101-000000_231231-234500.csv")
    selected_day = "2023-03-15"  # Wednesday; both directions have all 96 quarter-hours.
    coil = pd.read_csv(
        coil_path, sep=";", usecols=["datetime", "282_0", "282_1"],
        parse_dates=["datetime"], index_col="datetime"
    )
    daily = coil.loc[selected_day, ["282_0", "282_1"]]
    if len(daily) != 96 or daily.isna().any().any():
        raise ValueError(f"Incomplete 15-minute observations for coil 282 on {selected_day}")

    fig, ax = plt.subplots(figsize=(8.2, 4.8), dpi=300, constrained_layout=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    
    ax.plot(daily.index, daily["282_0"], color=PALETTE["roads_main"], lw=2.0, label="Direction 0")
    ax.plot(daily.index, daily["282_1"], color=PALETTE["roads_minor"], lw=2.0, label="Direction 1")
    ax.set_title("Coil 282 - Wednesday 15 March 2023")
    ax.set_xlabel("Time of day")
    ax.set_ylabel("Vehicles / 15 min")
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 3)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_xlim(daily.index.min(), daily.index.max())
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, ncol=2, loc="upper left")
    ax.grid(False)
    sns.despine(ax=ax)
    save_fig_local(fig, "1d_coil_profile")
    print(f"OK: plotted coil 282 on {selected_day}; 96 observations per direction")
except Exception as exc:
    print(f"ERROR 1d_coil_282_daily_profile: {type(exc).__name__}: {exc}")
    raise


OK: saved 1d_coil_profile.png and 1d_coil_profile.pdf in article/tex/images
OK: plotted coil 282 on 2023-03-15; 96 observations per direction


In [38]:
# coil_282_location.png - detailed location of coil 282 on a CartoDB Positron basemap.
try:
    import contextily as ctx
    zoom=10

    stations = pd.read_csv("data/er_barriers.csv", sep=";")
    station = stations.loc[stations["ID"] == 282].copy()
    if len(station) != 1:
        raise ValueError("Coil 282 must have exactly one coordinate record")

    point = gpd.GeoDataFrame(
        station,
        geometry=gpd.points_from_xy(station["Lon"], station["Lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=3857)
    x, y = point.geometry.iloc[0].x, point.geometry.iloc[0].y
    radius = 2800

    fig, ax = plt.subplots(figsize=(7.2, 7.2), dpi=300, constrained_layout=True)
    buffer = 30000

    ax.set_xlim(x - buffer, x + buffer)
    ax.set_ylim(y - buffer, y + buffer)
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        crs=point.crs,
        attribution=False,
        zoom=zoom,
    )
    point.plot(
        ax=ax, marker="o", color="#D55E00", edgecolor="white",
        linewidth=1.5, markersize=250, zorder=4,
    )
    ax.annotate(
        "Coil 282", xy=(x, y), xytext=(12, 14), textcoords="offset points",
        fontsize=30, fontweight="bold", color=PALETTE["text"],
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#DDDDDD", "alpha": 0.95},
        zorder=5,
    )
    # ax.set_title("Location of coil 282", fontsize=24, color=PALETTE["text"], pad=12)
    ax.set_axis_off()
    save_fig_local(fig, "coil_282_location")
    print(f"OK: coil 282 located at {station.iloc[0]['Lat']:.6f}, {station.iloc[0]['Lon']:.6f}")
except Exception as exc:
    print(f"ERROR coil_282_location: {type(exc).__name__}: {exc}")
    raise


OK: saved coil_282_location.png and coil_282_location.pdf in article/tex/images
OK: coil 282 located at 44.547614, 11.205961
